# 🤖 iTech Academy — AI Workshop
## Level 2 (Medium): AI Agents

---

> **You built a RAG system in Level 1.** Now we give the AI the power to *take actions* — search the web, do math, read files — on its own!

---

### 🧠 RAG vs Agent — What's the difference?

| | RAG (Level 1) | Agent (Level 2) |
|---|---|---|
| **What it does** | Reads your documents | Uses tools to find answers |
| **Can search web?** | ❌ No | ✅ Yes |
| **Can do math?** | ❌ No | ✅ Yes |
| **Thinks step by step?** | ❌ No | ✅ Yes |
| **Can use RAG too?** | — | ✅ Yes! |

---

### 🔄 How an Agent Thinks — The ReAct Loop

```
You ask: "What is 25% of the population of India?"

  🤔 THINK  → "I need the population of India first"
  🔧 ACT    → searches Wikipedia: "population of India"
  👀 OBSERVE → "1.4 billion"
  🤔 THINK  → "Now I calculate 25% of 1.4 billion"
  🔧 ACT    → uses calculator: 1400000000 × 0.25
  👀 OBSERVE → "350000000"
  ✅ ANSWER → "25% of India's population is 350 million"
```

### 🛠️ Tools We Use (All FREE)
| Tool | Purpose | Cost |
|------|---------|------|
| **Groq API** | Free LLM (Llama 3.1) | FREE |
| **LangChain** | Agent framework | FREE |
| **DuckDuckGo** | Free web search | FREE |
| **Wikipedia** | Free knowledge base | FREE |
| **ChromaDB** | Your RAG from Level 1 | FREE |

---
## 🔧 STEP 0 — Install & Import

In [1]:
!pip install -q groq langchain langchain-community langchain-groq duckduckgo-search wikipedia chromadb sentence-transformers

In [ ]:
import os
from groq import Groq

# 👇 Same key from Level 1
os.environ["GROQ_API_KEY"] = ""

client = Groq()
print("✅ Ready!")

✅ Ready!


---
## 🎯 DEMO 1 — The Problem: LLM Has No Tools

> Ask a simple question that needs **current info + math**. Watch the plain LLM struggle.

In [3]:
question = "What is today's date and what day of the week is it?"

resp = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": question}]
)

print("❌ Plain LLM (no tools):")
print(resp.choices[0].message.content)

❌ Plain LLM (no tools):
Unfortunately, I'm a large language model, I don't have real-time access to the current date and time. My knowledge cutoff is December 2023. However, I can suggest ways for you to find out the current date and day of the week.

You can check your device's calendar, search for the current date online, or use a voice assistant like Siri, Google Assistant, or Alexa to find out the date and day of the week.

If you want to know the date in March 2024, I can suggest that the 15th of March 2024 is a Friday,


In [4]:
# ✅ Now — give it a tool (just a Python function!)
from datetime import datetime

def get_date():
    return datetime.now().strftime("%A, %d %B %Y")

# Manually inject tool result — this is exactly what agents do automatically!
tool_result = get_date()

resp2 = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{
        "role": "user",
        "content": f"Tool result: today is {tool_result}.\n\nNow answer: {question}"
    }]
)

print("✅ LLM WITH a tool:")
print(resp2.choices[0].message.content)
print("\n💡 The agent does this automatically for every tool!")

✅ LLM WITH a tool:
Today's date is 15 March 2026, and today is a Sunday.

💡 The agent does this automatically for every tool!


---
## 🔧 SECTION 1 — Build Tools for Your Agent

> A **tool** is just a Python function with a description. That's it!
> The agent reads the description and decides WHEN to use it.

In [5]:
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

# ── Tool 1: Calculator ──────────────────────────────────────────
@tool
def calculator(expression: str) -> str:
    """Use this to do any math calculation. Input must be a valid Python math expression."""
    try:
        result = eval(expression)
        return f"{expression} = {result}"
    except:
        return "Could not calculate. Check your expression."

# ── Tool 2: Date & Time ─────────────────────────────────────────
@tool
def get_current_date(query: str) -> str:
    """Use this to get today's date or current time."""
    return f"Today is {datetime.now().strftime('%A, %d %B %Y, %H:%M')}"

# ── Tool 3: Wikipedia Search ────────────────────────────────────
import wikipedia

@tool
def wiki_search(query: str) -> str:
    """Search Wikipedia for factual information about any topic."""
    try:
        return wikipedia.summary(query, sentences=3)
    except:
        return f"No Wikipedia article found for '{query}'."

# ── Tool 4: Web Search (DuckDuckGo) ────────────────────────────
_ddg = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Search the web using DuckDuckGo. Use for current events or information not on Wikipedia."""
    result = _ddg.run(query)
    return result[:1500]  # Truncate to avoid overflowing the LLM context

# Collect all tools
tools = [calculator, get_current_date, wiki_search, web_search]

print(f"✅ {len(tools)} tools ready!")
for t in tools:
    print(f"  🔧 {t.name}: {t.description[:60]}...")

/Users/syedaskari/itech-workshop/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 4 tools ready!
  🔧 calculator: Use this to do any math calculation. Input must be a valid P...
  🔧 get_current_date: Use this to get today's date or current time....
  🔧 wiki_search: Search Wikipedia for factual information about any topic....
  🔧 web_search: Search the web using DuckDuckGo. Use for current events or i...


---
## 🤖 SECTION 2 — Build the Agent

> Only **5 lines** to create a full AI agent!

In [6]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
agent_executor = create_agent(llm, tools)
print("✅ Agent ready!")

✅ Agent ready!


---
## 🎯 DEMO 2 — Watch the Agent Think Step by Step!

> **`verbose=True`** shows EVERY thought the agent has.
> Watch for: **Thought → Action → Action Input → Observation**

In [7]:
# 🧪 Test 1: Simple Math
# Watch how it uses the calculator tool!

result = agent_executor.invoke({"messages": [("user", "What is 1234 multiplied by 5678?")]})
print("\n🏁 FINAL ANSWER:", result["messages"][-1].content)


🏁 FINAL ANSWER: The result of the calculation is 7006652.


In [8]:
# 🧪 Test 2: Multi-Step — needs TWO tools!
# Watch it use Wikipedia THEN calculator!

result = agent_executor.invoke({
    "messages": [("user", "What is the population of India? Calculate 10% of that number.")]
})
print("\n🏁 FINAL ANSWER:", result["messages"][-1].content)


🏁 FINAL ANSWER: The population of India is approximately 1.428 billion. 10% of that number is 0.1428 billion.


In [9]:
# 🧪 Test 3: Current World Info
# Watch it search the web!

result = agent_executor.invoke({
    "messages": [("user", "What is the latest version of Python programming language?")]
})
print("\n🏁 FINAL ANSWER:", result["messages"][-1].content)


🏁 FINAL ANSWER: The latest version of Python programming language is Python 3.14.


---
## 🎮 INTERACTIVE DEMO 3 — Ask the Agent ANYTHING!

> Change `my_question` and run. Try to **break it** or ask something tricky!

In [10]:
# 👇 Change this question and run the cell!
my_question = "What is today's date? How many days until New Year's Day 2027?"

result = agent_executor.invoke({"messages": [("user", my_question)]})
print("\n🏁 FINAL ANSWER:", result["messages"][-1].content)

KeyboardInterrupt: 

---
## 💡 SECTION 3 — Clean Output (Hide the Thinking)

> In a real app you don't want to show all the thinking steps.
> One line change: `verbose=False`

In [11]:
# Silent agent — clean output only
silent_agent = create_agent(llm, tools)

def ask(question):
    """Ask the agent a question and print only the answer."""
    result = silent_agent.invoke({"messages": [("user", question)]})
    print(f"❓ {question}")
    print(f"✅ {result['messages'][-1].content}\n")

# Quick fire questions!
ask("What is 99 multiplied by 99?")
ask("In one sentence: what is machine learning?")
ask("What is today's date?")

❓ What is 99 multiplied by 99?
✅ The result of 99 multiplied by 99 is 9801.



BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=web_search>{"query": "machine learning applications in finance" </function>'}}

---
## 🏆 LAB — RAG + Agent Combined!

> This is the **BOSS LEVEL**. We plug the RAG system from Level 1 **directly into the agent as a tool**.
> Now the agent can search the web AND search your documents!

```
Question about your doc?  → Agent uses RAG tool  → ChromaDB answers
Question about the world? → Agent uses web search → DuckDuckGo answers
Needs both?               → Agent uses BOTH!      → Best of both worlds 🔥
```

In [12]:
# Step 1: Rebuild the RAG system from Level 1 (quick version)
import chromadb
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

itech_doc = """
iTech Academy was founded in 2018 to train the next generation of tech professionals.
iTech Academy offers courses in AI, Web Development, Cybersecurity, and Data Science.
The AI Workshop is a 1-day intensive program covering RAG and AI Agents.
iTech Academy is located in Bengaluru, India.
Students who complete the AI Workshop receive an iTech Academy certificate.
The workshop uses 100% free and open source tools including Groq, LangChain, and ChromaDB.
iTech Academy workshops are designed for college students with basic Python knowledge.
"""

# Build vector store
chunks   = [s.strip() for s in itech_doc.strip().split("\n") if s.strip()]
embeddings = embed_model.encode(chunks).tolist()

db         = chromadb.Client()
collection = db.create_collection("itech")
collection.add(documents=chunks, embeddings=embeddings,
               ids=[f"c{i}" for i in range(len(chunks))])

print(f"✅ RAG ready — {len(chunks)} facts stored about iTech Academy")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5664.04it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ RAG ready — 7 facts stored about iTech Academy


In [13]:
# Step 2: Wrap the RAG system as an Agent Tool (just a function!)

@tool
def search_itech_docs(query: str) -> str:
    """Search iTech Academy's internal documents. Use this for questions about iTech Academy, its courses, workshops, location, and certificates."""
    q_emb   = embed_model.encode([query]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=2)
    return "\n".join(results["documents"][0])

# Step 3: Add RAG tool to the agent's toolbox
tools_with_rag = [calculator, get_current_date, wiki_search, web_search, search_itech_docs]

rag_agent = create_agent(llm, tools_with_rag)

print(f"✅ RAG-Agent ready with {len(tools_with_rag)} tools!")
for t in tools_with_rag:
    print(f"  🔧 {t.name}")

✅ RAG-Agent ready with 5 tools!
  🔧 calculator
  🔧 get_current_date
  🔧 wiki_search
  🔧 web_search
  🔧 search_itech_docs


In [14]:
# 🧪 Question about iTech docs — agent should use RAG tool
result = rag_agent.invoke({"messages": [("user", "Where is iTech Academy located and what courses do they offer?")]})
print("\n🏁 FINAL ANSWER:", result["messages"][-1].content)


🏁 FINAL ANSWER: The information about iTech Academy's location and courses is still unclear.


In [15]:
# 🧪 Question about the outside world — agent should use web search
result = rag_agent.invoke({"messages": [("user", "Who created the LangChain library?")]})
print("\n🏁 FINAL ANSWER:", result["messages"][-1].content)


🏁 FINAL ANSWER: The creator of the LangChain library is Harrison Chase.


In [16]:
# 🏆 BOSS QUESTION — needs BOTH tools!
result = rag_agent.invoke({
    "messages": [("user", "What AI tools does iTech Academy teach, and who made those tools?")]
})
print("\n🏁 FINAL ANSWER:", result["messages"][-1].content)

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01jq5sh544f6nt5b8phyxq016f` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 6221, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

---
## 🎮 MINI CHALLENGE — Add YOUR OWN Tool!

> Any Python function can become an agent tool in **3 lines**.
> Build your own and plug it in!

In [ ]:
# 👇 Create your own tool here — change the function!

@tool
def my_custom_tool(input: str) -> str:
    """Describe what your tool does here — the agent reads this to decide when to use it."""
    # 👇 Write your tool logic here
    return f"Your tool received: {input} — replace this with real logic!"

# IDEAS to try:
# - A tool that converts Celsius to Fahrenheit
# - A tool that counts words in a string
# - A tool that returns a random motivational quote
# - A tool that checks if a number is prime

# Add your tool to the agent
my_tools = tools_with_rag + [my_custom_tool]
my_agent = create_agent(llm, my_tools)

# Test your tool!
result = my_agent.invoke({"messages": [("user", "Use my_custom_tool with the input 'hello world'")]})
print("\n🏁 FINAL ANSWER:", result["messages"][-1].content)

---
## 🏁 FINAL RECAP

### ✅ What You Built Today (Both Levels)

```
Level 1 — RAG
  Your Document → Chunks → Embeddings → ChromaDB → Grounded Answers

Level 2 — Agent  
  Question → Think → Pick Tool → Act → Observe → Think → Answer

Level 2 Boss — RAG + Agent
  Question → Agent decides → RAG tool OR web search OR both → Perfect Answer 🔥
```

### 🧠 Key Concepts

| Concept | What it means |
|---|---|
| **ReAct Loop** | Think → Act → Observe → repeat until done |
| **Tool** | Any Python function the agent can call |
| **Agent Executor** | The engine that runs the Think→Act→Observe loop |
| **verbose=True** | Shows all the agent's internal thinking steps |
| **RAG Tool** | Your ChromaDB wrapped as a tool the agent can call |

### 🚀 What You Can Build Next
- 📚 A study assistant that reads your textbooks
- 🔍 A research bot that searches + summarises
- 💬 A customer support bot for any business
- 🧑‍💻 A coding assistant with access to documentation

---

> 🎉 **You just built a full AI Agent from scratch — using 100% free tools!**

---
*iTech Academy  |  AI Workshop 2025  |  All tools are 100% free & open source*